In [23]:
from langchain_text_splitters import MarkdownHeaderTextSplitter


def prepare_docs():
    # =============1.准备文档================
    with open('../day07/resources/评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    # 切分文档, 返回的是 list[Document]
    docs = markdown_splitter.split_text(doc)

    return docs

docs = prepare_docs()

In [24]:
from langchain_community.embeddings import DashScopeEmbeddings

# 1.初始化向量模型，可以把文档转稠密向量
embeddings = DashScopeEmbeddings(
    model="qwen3.7-text-embedding"
)

In [25]:
from app.core.config import settings
from pymilvus import MilvusClient
# 初始化Milvus客户端

client = MilvusClient(uri=settings.rag.milvus_url)

In [32]:
from pymilvus import DataType, Function, FunctionType

# 创建集合

# 准备集合
collection_name = "collection_name_1"
# 准备Schema
schema = client.create_schema(
    auto_id=False,  # 关闭主键自增
    enable_dynamic_field=False  # 关闭动态新增字段
)
# 准备普通字段
schema.add_field("id", datatype=DataType.INT64, is_primary=True)
schema.add_field("h2", datatype=DataType.VARCHAR, max_length=128)
schema.add_field("content", datatype=DataType.VARCHAR, max_length=2048, # 允许分词
                 enable_analyzer=True, andiyzer_params={"type": "chinese"}  # 指定分词器
                 )

# 准备向量字段
schema.add_field("dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
schema.add_field("sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
# 准备稀疏向量函数
sparse_function = Function(
    name="content_2_sparse",    # 函数名称， 保证唯一即可
    input_field_names=["content"],  # 待转换字段
    output_field_names=["sparse"],  # 待写入字段
    function_type=FunctionType.BM25 # 稀疏向量生成算法
)

schema.add_function(sparse_function)


# 为字段增加索引
index_params = client.prepare_index_params()
index_params.add_index(
    field_name="h2",    # 字段名称
    index_type="AUTOINDEX"  # 索引类型
)
index_params.add_index(
    field_name="dense",    # 字段名称
    index_type="AUTOINDEX",  # 索引类型
    metric_type="COSINE"    # 稠密向量相似度检索的算法，COSINE
)
index_params.add_index(
    field_name="sparse",    # 字段名称
    index_type="SPARSE_INVERTED_INDEX",  # 索引类型
    metric_type="BM25"    # 稀疏向量相似度检索的算法，BM25
)

# 创建集合
client.create_collection(
    collection_name=collection_name,
    schema=schema,
    index_params=index_params
)

In [27]:
# 查看Collection
result = client.list_collections()
print(result)   # ['collection']列表

['collection_name_1']


In [31]:
# 删除Collection
client.drop_collection(collection_name=collection_name)

In [33]:
# 插入数据 - client.insert()
data:list[dict] = []
for index, doc in enumerate(docs, start=1):
    # doc : Document对象
    entity = {
        "id": index,
        "h2": doc.metadata.get("h2", ""),
        "content": doc.page_content,
        "dense": embeddings.embed_query(doc.page_content)
    }
    data.append(entity)

res = client.insert(
    collection_name=collection_name,
    data=data
)
print(res)

{'insert_count': 21, 'ids': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]}


In [29]:
# 插入数据 - client.upsert()
# id为1的数据， 修改其h2字段， 模拟更新
data1 = data[0]
data1["h2"] = "测试更新"

# id为21的数据， 将其id修改为88， 模拟新增
data2 = data[20]
data2["id"] = 88

# 执行更新操作
res = client.upsert(collection_name, data=[data1, data2])
print(res)

In [30]:
# 根据id删除
res = client.delete(collection_name, ids=[1,88])
print(res)

{'delete_count': 2}
